In [5]:
import mysql.connector
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import getpass
password = getpass.getpass("Introduce la contraseña MySQL: ")

conn = mysql.connector.connect(
    host="212.227.90.6",
    user="Equipo26",
    password=password,
    database="Equip_26"
)

df_alojamientos = pd.read_sql("SELECT * FROM copy_ta27042026", conn)
conn.close()

print("Conexión exitosa ✅")
print(f"Datos cargados: {df_alojamientos.shape[0]} filas, {df_alojamientos.shape[1]} columnas")

/var/folders/vk/kwrzr2bd3ld451r5dkl52md80000gn/T/ipykernel_9907/1347343137.py:16: UserWarning:

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.



Conexión exitosa ✅
Datos cargados: 7693 filas, 37 columnas


In [6]:
df_alojamientos.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 7693 entries, 0 to 7692
Data columns (total 37 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   apartment_id                 7693 non-null   int64  
 1   name                         7690 non-null   object 
 2   description                  7643 non-null   object 
 3   host_id                      7693 non-null   int64  
 4   neighbourhood_name           7693 non-null   object 
 5   neighbourhood_district       4669 non-null   object 
 6   room_type                    7693 non-null   object 
 7   accommodates                 7693 non-null   int64  
 8   bathrooms                    7652 non-null   object 
 9   bedrooms                     7654 non-null   object 
 10  beds                         7685 non-null   float64
 11  amenities_list               7677 non-null   object 
 12  price                        7534 non-null   float64
 13  minimum_nights    

### Conclusiones: ###

Con la ampliació del Dataset, pasamos a tener una 7693 registros, de los cuales tenemos información de ratin general en 6059 de ellos. Por lo que se refiere a las valoraciones específicas, tenemos 6050 en precisión, 6054 en comunicación, 6056 en limpieza y 6045 en check-in.


In [7]:
df_alojamientos[df_alojamientos['review_scores_rating_clean'].notnull()]['apartment_id'].nunique()

6059

<span style="font-size: 1.5em;">Comprobamos de los registros en los que tenemos información, también existe informació en las valoraciones específicas</span>

In [8]:
# Filramos los registros que tienen valoración general y también información en las demás dimensiones de valoración
# 1. Definimos las columnas de dimensiones
dimensiones = [
    'review_scores_communication',
    'review_scores_checkin',
    'review_scores_cleanliness',
    'review_scores_accuracy'
]

# 2. Aplicamos el filtro correctamente
# Usamos .all(axis=1) para asegurar que TODAS las dimensiones sean no nulas
df_filtrado = df_alojamientos[
    (df_alojamientos['review_scores_rating_clean'].notnull()) & 
    (df_alojamientos[dimensiones].notnull().all(axis=1))
].copy()

# 3. Verificamos
# 2. Usamos el símbolo ~ (alt gr + 4) para seleccionar lo CONTRARIO al filtro
# Es decir: registros que tienen Rating pero les falta alguna dimensión
df_descartados = df_alojamientos[
    (df_alojamientos['review_scores_rating_clean'].notnull()) & 
    ~(df_alojamientos[dimensiones].notnull().all(axis=1))
].copy()

# 3. Mostramos los 16 registros "culpables"
print(f"Registros eliminados por información incompleta: {len(df_descartados)}")
display(df_descartados[['apartment_id', 'review_scores_rating_clean'] + dimensiones])


Registros eliminados por información incompleta: 16


,apartment_id,review_scores_rating_clean,review_scores_communication,review_scores_checkin,review_scores_cleanliness,review_scores_accuracy
709,3073348,20.0,80.0,20.0,100.0,NaN
1128,5486531,80.0,NaN,NaN,60.0,NaN
1415,7019190,100.0,100.0,NaN,100.0,100.0
1435,7113910,80.0,100.0,NaN,NaN,NaN
1476,7304217,100.0,100.0,NaN,NaN,NaN
1495,7396461,100.0,100.0,NaN,100.0,100.0
2372,13196950,80.0,80.0,NaN,NaN,NaN
2708,14162757,100.0,100.0,NaN,80.0,100.0
3209,16234816,60.0,80.0,60.0,40.0,NaN
3662,17649686,20.0,NaN,NaN,20.0,NaN


### Conclusiones: ###

Una vez filtramos el Dataset con la condición que tengo además de tener la valoración general también contengan información en los otros apartados de valoración. Una vez aplicado el filtro, pasamos de tener 6059 a 6043. Existían 16 registros que no tenían información en algunos de los campos de valoración a pesar de tener valoración general.

<span style="font-size: 1.5em;">Procedemos a realizar la segmentación entre alojamientos TOP y BOTTOM</span>

<span style="font-size: 1.2em;">Para hacer está segmentación y evitar que sea arbitraria, estudiaremos de que manera se distribuyen las valoraciones generales y en base a esto, estableceremos unos criterios objetivos</span>

In [9]:
# Calculamos los cuartiles Q1 y Q3
df_filtrado['review_scores_rating_clean'].describe()
q1_valor = df_filtrado['review_scores_rating_clean'].quantile(0.25)
q3_valor = df_filtrado['review_scores_rating_clean'].quantile(0.75)

### Conclusiones: ###

Los resultados nos indican que las valoraciones son elevadas, tal como indica que el 75% de las alojamientos tienen una valoración igual o superior a 89.
Otro dato signficado es el que un 50% de los alojamientos recibe una valoración igual o superior a 94 (superando la media 91.96, empujada hacia abajo por valoraciones extremadamente negativas).
En base a como se distribuyen las valoraciones generales, consideramos que un criterio objetivo para distinguir entre alojamientos buenos y malos seria el primer cuartil y el tercer cuartil

In [10]:
alojamientos_top = df_filtrado[df_filtrado['review_scores_rating_clean']>=q3_valor].copy()
print(alojamientos_top[dimensiones].info())

alojamientos_malos = df_filtrado[df_filtrado['review_scores_rating_clean']<=q1_valor].copy()
print(alojamientos_malos[dimensiones].info())

<class 'pandas.core.frame.DataFrame'>
Index: 1674 entries, 2 to 7691
Data columns (total 4 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   review_scores_communication  1674 non-null   float64
 1   review_scores_checkin        1674 non-null   float64
 2   review_scores_cleanliness    1674 non-null   float64
 3   review_scores_accuracy       1674 non-null   float64
dtypes: float64(4)
memory usage: 65.4 KB
None
<class 'pandas.core.frame.DataFrame'>
Index: 1572 entries, 6 to 7674
Data columns (total 4 columns):
 #   Column                       Non-Null Count  Dtype  
---  ------                       --------------  -----  
 0   review_scores_communication  1572 non-null   float64
 1   review_scores_checkin        1572 non-null   float64
 2   review_scores_cleanliness    1572 non-null   float64
 3   review_scores_accuracy       1572 non-null   float64
dtypes: float64(4)
memory usage: 61.4 KB
None


### Conclusiones: ###

Una vez realizamos la segmentación, tendremos un total de 1.674 registros que conforman el grupo de alojamientos top y 1.572 de alojamientos "malos", sumando un total de 3.246 observaciones.
Esto representa un total del 53,71 %, esto significa que no estamos comparando anécdotas o casos aislados, sino tendencias masivas del comportamiento de los clientes en la plataforma. De esta manera,
podemos afirmar que los resultados serán consistentes y representativos.

<span style="font-size: 1.5em;">Comparaciones valoraciones entre TOP y BOTTOM</span>

### Selección de Métricas de Comparación

In [11]:
alojamientos_malos[dimensiones].describe().round(2)

,review_scores_communication,review_scores_checkin,review_scores_cleanliness,review_scores_accuracy
count,1572.00,1572.00,1572.00,1572.00
mean,90.12,90.34,84.46,85.92
std,11.85,11.82,13.32,12.71
min,20.00,20.00,20.00,20.00
25%,90.00,90.00,80.00,80.00
50%,90.00,90.00,90.00,90.00
75%,100.00,100.00,90.00,90.00
max,100.00,100.00,100.00,100.00


In [12]:
alojamientos_top[dimensiones].describe().round(2)

,review_scores_communication,review_scores_checkin,review_scores_cleanliness,review_scores_accuracy
count,1674.00,1674.0,1674.00,1674.00
mean,99.26,99.0,98.32,99.03
std,4.18,4.7,5.21,4.24
min,20.00,40.0,40.00,60.00
25%,100.00,100.0,100.00,100.00
50%,100.00,100.0,100.00,100.00
75%,100.00,100.0,100.00,100.00
max,100.00,100.0,100.00,100.00


### Conclusiones:
Para este análisis se ha priorizado el uso de la media aritmética sobre la mediana, debido a su capacidad para integrar y reflejar de forma explícita el impacto de las valoraciones extremadamente bajas. En el grupo de bajo rendimiento, estas puntuaciones críticas representan fallos operativos severos que la mediana, por su naturaleza, tiende a neutralizar.

La elección de la media es la más adecuada para realizar un análisis de gaps (brechas) efectivo, ya que permite maximizar la visibilidad de las distancias entre el segmento de excelencia y el crítico.

In [13]:
medias_top = alojamientos_top.agg({'review_scores_communication':'mean', 'review_scores_checkin':'mean', 'review_scores_cleanliness':'mean', 'review_scores_accuracy':'mean'})
medias_malos = alojamientos_malos.agg({'review_scores_communication':'mean', 'review_scores_checkin':'mean', 'review_scores_cleanliness':'mean', 'review_scores_accuracy':'mean'})

gaps = pd.DataFrame({'Alojmientos excelentes':medias_top, 'Alojamientos bajo rendimiento': medias_malos, 'Diferencias': medias_top-medias_malos}).round(2).sort_values(by='Diferencias', ascending= False)
gaps

,Alojmientos excelentes,Alojamientos bajo rendimiento,Diferencias
review_scores_cleanliness,98.32,84.46,13.86
review_scores_accuracy,99.03,85.92,13.10
review_scores_communication,99.26,90.12,9.14
review_scores_checkin,99.00,90.34,8.65


### Conclusiones:
### Higiene (Limpieza): El Umbral de la Excelencia
### Diferencial: 13.86 puntos
La limpieza no es solo una métrica, es el pilar sobre el que se construye toda la experiencia. Los resultados demuestran que un alojamiento excelente no se distingue por detalles superficiales, sino por un estado impecable del espacio. Los alojamientos de bajo rendimiento fallan estrepitosamente en esta área, siendo esta la razón principal de su baja nota global. Mientras que el grupo de alto rendimiento roza la perfección, el segmento crítico muestra una degradación operativa que el cliente no perdona. La higiene es el factor "hace o deshace" (make or break) de este negocio.

### Precisión (Veracidad): El Contrato de Confianza
### Diferencial: 13.10 puntos
Este gap revela un problema de honestidad comercial. En el segmento de bajo rendimiento, las fotos y las descripciones actúan como una promesa incumplida; el cliente llega con una expectativa generada por el anuncio y se encuentra con una realidad decepcionante. No se trata solo de que el alojamiento sea mejor o peor, sino de que "miente". La excelencia en los alojamientos top reside en la coherencia total entre lo que se vende y lo que se entrega, eliminando cualquier fricción por falsas expectativas.

### Check-in: El Estándar Operativo Mínimo
### Diferencial: 8.65 puntos
El proceso de entrada es la dimensión con menor diferencia, lo que indica que es una capacidad ya "comoditizada" en el mercado. Incluso los peores alojamientos suelen tener un proceso de entrada aceptable. Esto nos dice que mejorar el check-in no es una palanca estratégica para pasar de ser "malo" a "excelente"; es simplemente un requisito logístico que casi todos los actores dominan. Un check-in fluido no garantiza una buena nota, pero se da por sentado como el mínimo profesional exigible.

### Comunicación: El Factor de Cortesía
### Diferencial: 9.14 puntos
Al igual que el check-in, la comunicación presenta una brecha reducida. Los anfitriones del segmento crítico suelen responder a los mensajes y mantener un trato básico adecuado, lo que demuestra que la "simpatía" o la rapidez de respuesta no compensan las deficiencias físicas del alojamiento. La comunicación se da por supuesta y no actúa como un elemento diferenciador; es una condición necesaria pero no suficiente para el éxito.

* GRAFICOS:

In [14]:
import plotly.graph_objects as go

dimensiones = {
    'Higiene':      df_alojamientos['review_scores_cleanliness'].mean(),
    'Precisión':    df_alojamientos['review_scores_accuracy'].mean(),
    'Check-in':     df_alojamientos['review_scores_checkin'].mean(),
    'Comunicación': df_alojamientos['review_scores_communication'].mean(),
}

dimensiones_sorted = dict(sorted(dimensiones.items(), key=lambda x: x[1], reverse=False))
valores = list(dimensiones_sorted.values())
etiquetas = list(dimensiones_sorted.keys())

# Gradiente azul uniforme — oscuro abajo, claro arriba
# Check-in y Comunicación empatan → mismo color claro
colores = ['#B8DDE4', '#2C7BB6', '#1A4E6E', '#1A4E6E']

fig_dimensiones = go.Figure(go.Bar(
    x=valores,
    y=etiquetas,
    orientation='h',
    marker_color=colores,
    marker_line_width=0,
    text=[round(v, 1) for v in valores],
    textposition='inside',
    textfont=dict(color='white', size=16, family='Arial'),
))

fig_dimensiones.update_layout(
    title=dict(
        text='¿Cómo valoran los huéspedes los aspectos clave del servicio?<br><sup>Puntuación media global (0-100) — Precisión, Higiene, Check-in y Comunicación</sup>',
        font=dict(size=20, family='Arial', color='#1a1a1a'),
        x=0.5,
        xanchor='center'
    ),
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(visible=False, range=[90, 98]),
    yaxis=dict(
        showgrid=False,
        tickfont=dict(size=18, family='Arial', color='#1a1a1a'),
        ticksuffix='  '
    ),
    bargap=0.4,
    showlegend=False,
    margin=dict(l=20, r=20, t=100, b=20),
    height=400
)

fig_dimensiones.show()

In [15]:
fig_top_bottom = go.Figure()

etiquetas_ord = ['Check-in', 'Comunicación', 'Precisión', 'Limpieza']
medias_top_ord = [99.0, 99.3, 99.0, 98.3]
medias_bottom_ord = [90.3, 90.1, 85.9, 84.5]
diferencias = [round(t - b, 1) for t, b in zip(medias_top_ord, medias_bottom_ord)]

fig_top_bottom.add_trace(go.Bar(
    x=medias_top_ord,
    y=etiquetas_ord,
    orientation='h',
    name='Alojamientos excelentes',
    marker_color='#2C7BB6',
    marker_line_width=0,
    text=medias_top_ord,
    textposition='inside',
    textfont=dict(color='white', size=15, family='Arial'),
))

fig_top_bottom.add_trace(go.Bar(
    x=medias_bottom_ord,
    y=etiquetas_ord,
    orientation='h',
    name='Alojamientos de bajo rendimiento',
    marker_color='lightcoral',
    marker_line_width=0,
    text=medias_bottom_ord,
    textposition='inside',
    textfont=dict(color='white', size=15, family='Arial'),
))

# Anotaciones de diferencia
for i, (dif, top) in enumerate(zip(diferencias, medias_top_ord)):
    fig_top_bottom.add_annotation(
        x=top + 0.5,
        y=i,
        text=f'Δ {dif} pts',
        showarrow=False,
        font=dict(size=14, family='Arial', color='#1A4E6E'),
        xanchor='left',
        yshift=10
    )

fig_top_bottom.update_layout(
    title=dict(
        text='¿Qué aspectos diferencian los alojamientos mejor y peor valorados?<br><sup>Puntuación media (0-100) por dimensión — Alojamientos excelentes vs. bajo rendimiento</sup>',
        font=dict(size=20, family='Arial', color='#1a1a1a'),
        x=0.5,
        xanchor='center'
    ),
    barmode='group',
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(visible=False, range=[75, 110]),
    yaxis=dict(
        showgrid=False,
        tickfont=dict(size=16, family='Arial', color='#1a1a1a'),
        ticksuffix='  '
    ),
    bargap=0.3,
    bargroupgap=0.1,
    legend=dict(
        font=dict(size=14, family='Arial', color='#1a1a1a'),
        x=0.5,
        y=-0.15,
        orientation='h',
        xanchor='center'
    ),
    margin=dict(l=20, r=60, t=100, b=80),
    height=420
)

fig_top_bottom.show()

In [16]:
import statsmodels.api as sm

# 1. Definimos las variables independientes (X) y la dependiente (y)
# Usamos el dataframe filtrado original (los 6043 registros) para tener toda la variabilidad
X = df_filtrado[['review_scores_cleanliness', 'review_scores_accuracy', 
                 'review_scores_communication', 'review_scores_checkin']]
y = df_filtrado['review_scores_rating_clean']

# 2. Añadimos una constante (intercepto) al modelo
X = sm.add_constant(X)

# 3. Ajustamos el modelo
modelo = sm.OLS(y, X).fit()

# 4. Mostramos el resumen
print(modelo.summary())

                                OLS Regression Results                                
Dep. Variable:     review_scores_rating_clean   R-squared:                       0.667
Model:                                    OLS   Adj. R-squared:                  0.667
Method:                         Least Squares   F-statistic:                     3021.
Date:                        Sun, 03 May 2026   Prob (F-statistic):               0.00
Time:                                16:37:31   Log-Likelihood:                -18441.
No. Observations:                        6043   AIC:                         3.689e+04
Df Residuals:                            6038   BIC:                         3.692e+04
Df Model:                                   4                                         
Covariance Type:                    nonrobust                                         
                                  coef    std err          t      P>|t|      [0.025      0.975]
----------------------------------

In [17]:
import plotly.graph_objects as go

etiquetas_reg = ['Check-in', 'Comunicación', 'Limpieza', 'Precisión']
coefs = [0.086, 0.186, 0.298, 0.376]
coefs_redondeados = [round(c, 2) for c in coefs]
colores = ['#B8DDE4', '#74C2C9', '#2C7BB6', '#1A4E6E']

fig_regresion = go.Figure(go.Bar(
    x=coefs,
    y=etiquetas_reg,
    orientation='h',
    marker_color=colores,
    marker_line_width=0,
    text=coefs_redondeados,
    textposition='inside',
    textfont=dict(color='white', size=15, family='Arial'),
))

fig_regresion.update_layout(
    title=dict(
        text='¿Qué dimensiones impactan más en el rating general?<br><sup>Regresión lineal — R² = 0.667 | Por cada punto de mejora en la dimensión, el rating sube X puntos</sup>',
        font=dict(size=20, family='Arial', color='#1a1a1a'),
        x=0.5,
        xanchor='center'
    ),
    annotations=[
        dict(
            text='Ej: si la Precisión del anuncio sube 1 punto → el rating general sube 0.38 puntos',
            xref='paper', yref='paper',
            x=0.5, y=-0.15,
            showarrow=False,
            font=dict(size=16, family='Arial', color='#555555'),
            xanchor='center'
        )
    ],
    plot_bgcolor='white',
    paper_bgcolor='white',
    xaxis=dict(visible=False, range=[0, 0.45]),
    yaxis=dict(
        showgrid=False,
        tickfont=dict(size=16, family='Arial', color='#1a1a1a'),
        ticksuffix='  '
    ),
    bargap=0.4,
    showlegend=False,
    margin=dict(l=20, r=20, t=120, b=60),
    height=400
)

fig_regresion.show()

In [18]:
# Exportar gráficos en alta calidad para Power BI
fig_dimensiones.write_image("fig_dimensiones_global.png", scale=3, width=1200, height=400)
fig_top_bottom.write_image("fig_top_bottom.png", scale=3, width=1200, height=450)
fig_regresion.write_image("fig_regresion.png", scale=3, width=1200, height=400)

print("Gráficos exportados ✅")

Gráficos exportados ✅
